# 30_meta_1d_mass_rl2_seqppo

Düzeltilmiş RL² + sekans tabanlı PPO (TBPTT tarzı) ile 1D kütle ortamı.

- Ortam: 1D point-mass, kütle m değişiyor, sürtünme b ve hedef x_goal sabit.
- Ajan: GRU tabanlı RL² policy (augmented observation: [x, v, a_prev, r_prev, d_prev]).
- PPO: Sekans tabanlı, zaman sırasını bozmadan ve GRU'yu stateful kullanarak güncelleme.
- Truncated BPTT: Uzun trajectory, sabit uzunluklu segmentlere bölünüp her segmentte ayrı backward/step.


In [1]:
import os
import random
import warnings
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == 'cuda':
    torch.cuda.manual_seed_all(SEED)


Device: mps


## Hiperparametreler ve ortam ayarları


In [2]:
# Meta-RL (RL²) hiperparametreleri
META_BATCH_SIZE   = 8      # her meta-iterasyonda kaç farklı task (farklı m)
EPISODES_PER_TASK = 4      # her task için ardışık epizot sayısı (aynı GRU hidden state)
ROLLOUT_LEN       = 100    # her epizotta maksimum step
TOTAL_ITERS       = 200    # meta-iterasyon sayısı

GAMMA       = 0.99
LAMBDA_GAE  = 0.95
PPO_EPOCHS  = 4
PPO_CLIP    = 0.2
PPO_LR      = 3e-4
HIDDEN_SIZE = 64

# Sekans tabanlı PPO (TBPTT) için segment uzunluğu
SEG_LEN = 32   # truncated BPTT segment uzunluğu (step sayısı)

# Ortam parametreleri
DT          = 0.05
MAX_STEPS   = ROLLOUT_LEN
ACTION_LOW  = -2.0
ACTION_HIGH =  2.0

# Task dağılımı: sadece kütle m değişiyor; b ve x_goal sabit
M_VALUES   = [1.0, 2.0, 3.0]
B_FIXED    = 0.3
X_GOAL_FIX = 2.0


## 1D kütle ortamı ve task tanımı (sadece m değişiyor)


In [3]:
@dataclass
class MassTask:
    m: float
    b: float
    x_goal: float

def sample_task():
    m = random.choice(M_VALUES)
    b = B_FIXED
    xg = X_GOAL_FIX
    return MassTask(m, b, xg)

class OneDMassEnv:
    def __init__(self, task: MassTask):
        self.task = task
        self.dt   = DT
        self.max_steps = MAX_STEPS
        self.action_low  = ACTION_LOW
        self.action_high = ACTION_HIGH
        self.t = 0
        self.x = 0.0
        self.v = 0.0

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.x = np.random.uniform(-0.5, 0.5)
        self.v = np.random.uniform(-0.5, 0.5)
        self.t = 0
        obs = np.array([self.x, self.v], dtype=np.float32)
        return obs, {}
    

    def step(self, u):
        u = float(np.clip(u, self.action_low, self.action_high))
        m = self.task.m
        b = self.task.b
        xg = self.task.x_goal

        # Dinamik: m * dv/dt = u - b * v
        a = (u - b * self.v) / m
        self.v = self.v + a * self.dt
        self.x = self.x + self.v * self.dt
        self.t += 1

        dist2 = (self.x - xg) ** 2
        r = -dist2

        done = self.t >= self.max_steps
        trunc = False
        obs = np.array([self.x, self.v], dtype=np.float32)
        info = {'task': self.task}
        return obs, float(r), bool(done), bool(trunc), info
    def close(self):
        # Gym uyumluluğu için boş bırakıyoruz
        pass

print('Örnek task:', sample_task())
env_dbg = OneDMassEnv(sample_task())
s_dbg, _ = env_dbg.reset()
print('Initial state:', s_dbg, '| task:', env_dbg.task)


Örnek task: MassTask(m=3.0, b=0.3, x_goal=2.0)
Initial state: [-0.12545988  0.45071432] | task: MassTask(m=1.0, b=0.3, x_goal=2.0)


## RL² girdi yapısı: augmented observation


In [4]:
def make_initial_history(action_dim: int):
    a_prev = np.zeros((action_dim,), dtype=np.float32)
    r_prev = np.array([0.0], dtype=np.float32)
    d_prev = np.array([0.0], dtype=np.float32)
    return a_prev, r_prev, d_prev

def augment_obs(obs, a_prev, r_prev, d_prev):
    return np.concatenate([obs, a_prev, r_prev, d_prev], axis=-1).astype(np.float32)

test_task = sample_task()
test_env  = OneDMassEnv(test_task)
s0, _ = test_env.reset(seed=SEED)
a_prev, r_prev, d_prev = make_initial_history(action_dim=1)
aug0 = augment_obs(s0, a_prev, r_prev, d_prev)
print('Augmented obs shape:', aug0.shape)


Augmented obs shape: (5,)


## RL² policy: GRU tabanlı policy + value ağı


In [5]:
class RL2RecurrentPolicy(nn.Module):
    def __init__(self, input_dim: int, action_dim: int, hidden_size: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.action_dim  = action_dim

        self.fc_in = nn.Linear(input_dim, hidden_size)
        self.gru   = nn.GRU(hidden_size, hidden_size, batch_first=True)

        self.fc_pi   = nn.Linear(hidden_size, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.fc_v    = nn.Linear(hidden_size, 1)

    def initial_hidden(self, batch_size: int = 1, device: str = 'cpu'):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

    def forward_step(self, x_t: torch.Tensor, h: torch.Tensor):
        # x_t: (B, input_dim), h: (1, B, H)
        z = torch.relu(self.fc_in(x_t))
        z = z.unsqueeze(1)              # (B, 1, H)
        out, h_next = self.gru(z, h)    # out: (B, 1, H)
        out = out.squeeze(1)            # (B, H)

        mean = self.fc_pi(out)
        std  = self.log_std.exp().unsqueeze(0)
        dist = Normal(mean, std)
        action = dist.sample()
        logp   = dist.log_prob(action).sum(dim=-1)
        value  = self.fc_v(out).squeeze(-1)
        return action, logp, value, h_next

    def act_deterministic(self, x_t: torch.Tensor, h: torch.Tensor):
        z = torch.relu(self.fc_in(x_t))
        z = z.unsqueeze(1)
        out, h_next = self.gru(z, h)
        out = out.squeeze(1)
        mean  = self.fc_pi(out)
        value = self.fc_v(out).squeeze(-1)
        return mean, value, h_next

policy_dbg = RL2RecurrentPolicy(input_dim=aug0.shape[0], action_dim=1, hidden_size=HIDDEN_SIZE).to(device)
h0_dbg = policy_dbg.initial_hidden(batch_size=1, device=device)
x_dbg = torch.tensor(aug0, dtype=torch.float32, device=device).unsqueeze(0)
a_dbg, logp_dbg, v_dbg, h1_dbg = policy_dbg.forward_step(x_dbg, h0_dbg)
print('Policy debug action/val/logp:', a_dbg.detach().cpu().numpy(), v_dbg.item(), logp_dbg.item())


Policy debug action/val/logp: [[0.8982889]] 0.06070079654455185 -1.3281594514846802


## RL² Ajanı: collect_batch + sekans tabanlı PPO (TBPTT)


In [6]:
class RL2Agent:
    def __init__(self, input_dim, action_dim, hidden_size=HIDDEN_SIZE, lr=PPO_LR, device=device):
        self.device = device
        self.policy = RL2RecurrentPolicy(input_dim, action_dim, hidden_size).to(device)
        self.optim  = torch.optim.Adam(self.policy.parameters(), lr=lr)

    def collect_batch(self,
                       meta_batch_size=META_BATCH_SIZE,
                       episodes_per_task=EPISODES_PER_TASK,
                       rollout_len=ROLLOUT_LEN):
        obs_buf, act_buf, logp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []
        z_buf = []
        m_buf = []
        task_start_buf = []

        ep_returns_per_task = []

        for task_idx in range(meta_batch_size):
            task = sample_task()
            env  = OneDMassEnv(task)
            action_dim = 1

            h = self.policy.initial_hidden(batch_size=1, device=self.device)
            first_step_of_task = True

            task_ep_returns = []
            for ep in range(episodes_per_task):
                s, _ = env.reset(seed=SEED + task_idx * 100 + ep)
                prev_a, prev_r, prev_d = make_initial_history(action_dim)
                ep_ret = 0.0

                for t in range(rollout_len):
                    aug = augment_obs(s, prev_a, prev_r, prev_d)
                    x_t = torch.tensor(aug, dtype=torch.float32, device=self.device).unsqueeze(0)
                    with torch.no_grad():
                        a_t, logp_t, v_t, h = self.policy.forward_step(x_t, h)
                    a_np = a_t.squeeze(0).cpu().numpy()
                    ns, r, done, trunc, info = env.step(a_np[0])

                    if first_step_of_task:
                        task_start_buf.append(1.0)
                        first_step_of_task = False
                    else:
                        task_start_buf.append(0.0)

                    obs_buf.append(aug)
                    act_buf.append(a_np)
                    logp_buf.append(logp_t.cpu().numpy())
                    rew_buf.append(r)
                    val_buf.append(v_t.cpu().numpy())
                    done_flag = float(done or trunc)
                    done_buf.append(done_flag)

                    z_buf.append(h.squeeze(0).squeeze(0).detach().cpu().numpy())
                    m_buf.append(task.m)

                    ep_ret += r

                    prev_a = np.array([a_np[0]], dtype=np.float32)
                    prev_r = np.array([r], dtype=np.float32)
                    prev_d = np.array([done_flag], dtype=np.float32)
                    s = ns
                    if done or trunc:
                        break

                task_ep_returns.append(ep_ret)

            ep_returns_per_task.append(task_ep_returns)
            env.close()

        batch = {
            'obs':        torch.tensor(np.stack(obs_buf), dtype=torch.float32, device=self.device),
            'act':        torch.tensor(np.stack(act_buf), dtype=torch.float32, device=self.device),
            'logp':       torch.tensor(np.stack(logp_buf).squeeze(-1), dtype=torch.float32, device=self.device),
            'rew':        torch.tensor(np.array(rew_buf), dtype=torch.float32, device=self.device),
            'val':        torch.tensor(np.stack(val_buf).squeeze(-1), dtype=torch.float32, device=self.device),
            'done':       torch.tensor(np.array(done_buf), dtype=torch.float32, device=self.device),
            'z':          torch.tensor(np.stack(z_buf), dtype=torch.float32, device=self.device),
            'm':          torch.tensor(np.array(m_buf), dtype=torch.float32, device=self.device),
            'task_start': torch.tensor(np.array(task_start_buf), dtype=torch.float32, device=self.device)
        }

        return batch, ep_returns_per_task

    def compute_gae(self, rew, val, done, gamma=GAMMA, lam=LAMBDA_GAE):
        T = len(rew)
        adv = torch.zeros(T, dtype=torch.float32, device=rew.device)
        last_gae = 0.0
        for t in reversed(range(T)):
            nonterminal = 1.0 - done[t]
            if t == T - 1:
                next_value = 0.0
            else:
                next_value = val[t + 1]
            delta = rew[t] + gamma * next_value * nonterminal - val[t]
            last_gae = delta + gamma * lam * nonterminal * last_gae
            adv[t] = last_gae
        ret = adv + val
        return adv, ret

    def ppo_update(self, batch, epochs=PPO_EPOCHS, clip_ratio=PPO_CLIP, seg_len=SEG_LEN):
        obs        = batch['obs']
        act        = batch['act']
        old_logp   = batch['logp']
        rew        = batch['rew']
        val        = batch['val']
        done       = batch['done']
        task_start = batch['task_start']

        adv, ret = self.compute_gae(rew, val, done)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        T = obs.shape[0]

        last_pi_loss = 0.0
        last_v_loss  = 0.0

        for ep in range(epochs):
            h = self.policy.initial_hidden(batch_size=1, device=self.device)
            t = 0
            while t < T:
                seg_start = t
                seg_end   = min(t + seg_len, T)

                self.optim.zero_grad()
                seg_policy_losses = []
                seg_value_losses  = []
                seg_entropies     = []

                while t < seg_end:
                    if task_start[t] > 0.5:
                        h = self.policy.initial_hidden(batch_size=1, device=self.device)

                    x_t        = obs[t].unsqueeze(0)
                    a_t        = act[t].unsqueeze(0)
                    adv_t      = adv[t].unsqueeze(0)
                    ret_t      = ret[t].unsqueeze(0)
                    old_logp_t = old_logp[t].unsqueeze(0)

                    mean_t, v_t, h = self.policy.act_deterministic(x_t, h)
                    dist  = Normal(mean_t, self.policy.log_std.exp().unsqueeze(0))
                    logp_t = dist.log_prob(a_t).sum(dim=-1)

                    ratio = torch.exp(logp_t - old_logp_t)
                    surr1 = ratio * adv_t
                    surr2 = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_t
                    policy_loss_t = -torch.min(surr1, surr2).mean()
                    value_loss_t  = F.mse_loss(v_t.squeeze(-1), ret_t)
                    entropy_t     = dist.entropy().sum(dim=-1).mean()

                    seg_policy_losses.append(policy_loss_t)
                    seg_value_losses.append(value_loss_t)
                    seg_entropies.append(entropy_t)

                    t += 1

                if len(seg_policy_losses) == 0:
                    break

                policy_loss = torch.stack(seg_policy_losses).mean()
                value_loss  = torch.stack(seg_value_losses).mean()
                entropy     = torch.stack(seg_entropies).mean()

                loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
                self.optim.step()

                h = h.detach()

                last_pi_loss = float(policy_loss.item())
                last_v_loss  = float(value_loss.item())

        return last_pi_loss, last_v_loss

input_dim  = 2 + 1 + 1 + 1
action_dim = 1
agent = RL2Agent(input_dim=input_dim, action_dim=action_dim, hidden_size=HIDDEN_SIZE, lr=PPO_LR, device=device)
print('RL2Agent hazır.')


RL2Agent hazır.


## Eğitim döngüsü


In [7]:
avg_return_history = []
last_ep_return_history = []
pi_loss_history = []
v_loss_history  = []

z_all = []
m_all = []

for it in range(1, TOTAL_ITERS + 1):
    batch, ep_returns_per_task = agent.collect_batch()
    pi_loss, v_loss = agent.ppo_update(batch)

    # --- Güvenli ortalama hesaplama ---
    flat_returns = []
    last_eps = []
    for eps in ep_returns_per_task:
        if len(eps) > 0:
            flat_returns.extend(eps)   # tüm epizotların return'ları
            last_eps.append(eps[-1])  # her task için son epizot return'u

    if len(flat_returns) > 0:
        avg_ret = float(np.mean(flat_returns))
    else:
        avg_ret = 0.0

    if len(last_eps) > 0:
        last_ep_ret = float(np.mean(last_eps))
    else:
        last_ep_ret = 0.0

    avg_return_history.append(avg_ret)
    last_ep_return_history.append(last_ep_ret)
    pi_loss_history.append(pi_loss)
    v_loss_history.append(v_loss)

    # --- z ve m gerçekten varsa kaydet ---
    if 'z' in batch and 'm' in batch:
        z_all.append(batch['z'].detach().cpu().numpy())
        m_all.append(batch['m'].detach().cpu().numpy())

    if it % 10 == 0:
        print(
            f'Iter {it:04d} | '
            f'avg_ret={avg_ret:.3f} | '
            f'last_ep_ret={last_ep_ret:.3f} | '
            f'pi_loss={pi_loss:.3f} | v_loss={v_loss:.3f}'
        )

# --- t-SNE için latent durumları ancak veri varsa birleştir ---
if len(z_all) > 0:
    z_all = np.concatenate(z_all, axis=0)
    m_all = np.concatenate(m_all, axis=0)
    print('Toplam z_all shape:', z_all.shape)
else:
    print('z_all boş, henüz latent state kaydı yok (t-SNE için veri toplanmadı).')

Iter 0010 | avg_ret=-288.123 | last_ep_ret=-271.021 | pi_loss=-0.754 | v_loss=8.785
Iter 0020 | avg_ret=-204.656 | last_ep_ret=-191.462 | pi_loss=-0.666 | v_loss=60.501
Iter 0030 | avg_ret=-156.837 | last_ep_ret=-124.274 | pi_loss=-0.356 | v_loss=4.178
Iter 0040 | avg_ret=-118.405 | last_ep_ret=-103.704 | pi_loss=-0.254 | v_loss=1.578
Iter 0050 | avg_ret=-110.769 | last_ep_ret=-134.682 | pi_loss=0.009 | v_loss=1.255
Iter 0060 | avg_ret=-200.135 | last_ep_ret=-127.871 | pi_loss=-0.197 | v_loss=0.706
Iter 0070 | avg_ret=-1487.767 | last_ep_ret=-1575.176 | pi_loss=-0.623 | v_loss=3853.445
Iter 0080 | avg_ret=-2093.688 | last_ep_ret=-2179.668 | pi_loss=1.648 | v_loss=1007801.500
Iter 0090 | avg_ret=-1769.488 | last_ep_ret=-1704.169 | pi_loss=-0.241 | v_loss=27132.625
Iter 0100 | avg_ret=-1676.516 | last_ep_ret=-1634.736 | pi_loss=-0.782 | v_loss=17992.473
Iter 0110 | avg_ret=-2276.307 | last_ep_ret=-2301.380 | pi_loss=1.771 | v_loss=1065507.750


KeyboardInterrupt: 

## Öğrenme eğrileri


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(avg_return_history, label='avg return (tüm epizotlar)')
plt.plot(last_ep_return_history, label='last-episode return (task başına)')
plt.xlabel('Meta-iterasyon')
plt.ylabel('Return')
plt.title('Öğrenme eğrisi')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(pi_loss_history, label='policy loss')
plt.plot(v_loss_history, label='value loss')
plt.xlabel('Meta-iterasyon')
plt.ylabel('Loss')
plt.title('PPO loss değerleri')
plt.legend()
plt.grid(True)
plt.show()
